# 04 — Faz 4: Tuning + Eşik + SHAP (regularize + AP-monitored early stopping)

Regularize arama (ön-işleme bir kez hesaplanır, döngüde yalnız LGBM) + **AP-izleyen early stopping**
+ **gap hard-guardrail (≤0.10)**. `seller_rate` final config'te yeniden doğrulanır (tek iç val-split).
Eşik: varsayım(A recall-öncelikli) + PR eğrisi + OP1/OP2 + top-K. TargetEncoder Pipeline içinde. MLflow(sqlite).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json,numpy as np,pandas as pd,joblib
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler,StandardScaler,TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score,roc_auc_score,precision_recall_curve,
    precision_score,recall_score,f1_score)
from lightgbm import LGBMClassifier,early_stopping
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from rebuild import rebuild,TARGET
SEED=42; np.random.seed(SEED); print("setup ok")

setup ok


In [2]:
ROBUST=["variation_count","offer_count_trend","Package: Weight (g)"]
STANDARD=["Reviews: Rating","new_price_log","review_count_log","sr_log","review_velocity",
          "new_price_margin_est","sr_drops_90","Package: Dimension (cm³)"]
PASS=["has_sales_data","is_negative_margin","product_age_segment","is_active_seller"]
FREQ=["Categories: Sub","Brand"]; TGT_SEL=["listing_seller"]
def feat_names(us): return ROBUST+STANDARD+[c+"__freq" for c in FREQ]+PASS+([c+"__sellerrate" for c in TGT_SEL] if us else [])
rb=rebuild(); y=rb[TARGET].astype(int); BASE=float(y.mean())
X=rb[ROBUST+STANDARD+PASS+FREQ+TGT_SEL].copy()
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
print(f"base={BASE:.4f} | train={len(X_tr):,} | holdout={len(X_te):,}")

base=0.1457 | train=46,626 | holdout=8,229


In [3]:
class FrequencyEncoder(BaseEstimator,TransformerMixin):
    def __init__(self,min_count=50): self.min_count=min_count
    def fit(self,X,y=None):
        X=pd.DataFrame(X); self.maps_,self.rare_={},{}
        for c in X.columns:
            f=X[c].value_counts(); rare=set(f[f<self.min_count].index)
            coll=X[c].where(~X[c].isin(rare),"other")
            self.maps_[c]=coll.value_counts(normalize=True).to_dict(); self.rare_[c]=rare
        return self
    def transform(self,X):
        X=pd.DataFrame(X); out=np.zeros((len(X),X.shape[1]))
        for j,c in enumerate(X.columns):
            coll=X[c].where(~X[c].isin(self.rare_[c]),"other"); out[:,j]=coll.map(self.maps_[c]).fillna(0).values
        return out
def build_pre(us):
    s=[("robust",RobustScaler(),ROBUST),("standard",StandardScaler(),STANDARD),
       ("freq",FrequencyEncoder(50),FREQ),("pass","passthrough",PASS)]
    if us: s.append(("sellerrate",TargetEncoder(target_type="binary",cv=3,random_state=SEED),TGT_SEL))
    return ColumnTransformer(s,remainder="drop")
def ap_eval(yt,yp): return "AP",average_precision_score(yt,yp),True
def lgbm(p,n,spw): return LGBMClassifier(n_estimators=n,scale_pos_weight=spw,random_state=SEED,
        n_jobs=1,verbose=-1,metric="None",first_metric_only=True,**p)
print("helpers ready")

helpers ready


In [4]:
# ARAMA: on-isleme bir kez (urun-only); donguda yalniz LGBM -> hizli
Xi_s,Xe_s,yi_s,ye_s=train_test_split(X_tr,y_tr,test_size=0.2,stratify=y_tr,random_state=SEED)
pre_s=build_pre(False).fit(Xi_s,yi_s); Zi_s=pre_s.transform(Xi_s); Ze_s=pre_s.transform(Xe_s)
spw_s=float((yi_s==0).sum()/(yi_s==1).sum())
def objective(trial):
    p=dict(learning_rate=trial.suggest_float("learning_rate",0.02,0.07,log=True),
        num_leaves=trial.suggest_int("num_leaves",8,20),
        min_child_samples=trial.suggest_int("min_child_samples",100,300),
        reg_lambda=trial.suggest_float("reg_lambda",1.0,12.0),
        reg_alpha=trial.suggest_float("reg_alpha",0.0,5.0),
        subsample=trial.suggest_float("subsample",0.6,0.95),
        colsample_bytree=trial.suggest_float("colsample_bytree",0.6,0.95))
    m=lgbm(p,300,spw_s); m.fit(Zi_s,yi_s,eval_set=[(Ze_s,ye_s)],eval_metric=ap_eval,
        callbacks=[early_stopping(30,verbose=False)])
    val=average_precision_score(ye_s,m.predict_proba(Ze_s)[:,1])
    tr=average_precision_score(yi_s,m.predict_proba(Zi_s)[:,1]); trial.set_user_attr("gap",tr-val)
    return val
study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective,n_trials=12,show_progress_bar=False)
done=[t for t in study.trials if t.value is not None]
ok=[t for t in done if t.user_attrs.get("gap",1)<=0.10]; pool=ok if ok else done
bt=max(pool,key=lambda t:t.value); best=bt.params
print(f"guardrail-gecen: {len(ok)}/{len(done)} | secilen val PR-AUC={bt.value:.4f} gap={bt.user_attrs['gap']:+.4f}")
print("best:",best)

guardrail-gecen: 1/12 | secilen val PR-AUC=0.5759 gap=+0.0976
best: {'learning_rate': 0.042009143939033815, 'num_leaves': 8, 'min_child_samples': 222, 'reg_lambda': 2.875765360560207, 'reg_alpha': 0.3252579649263976, 'subsample': 0.9321099380386666, 'colsample_bytree': 0.9379712115760958}


In [5]:
# seller reval + final: tek ic val-split (tek cekirdek icin hizli)
X2,Xv,y2,yv=train_test_split(X_tr,y_tr,test_size=0.2,stratify=y_tr,random_state=SEED)
def eval_variant(us):
    pre=build_pre(us); Z2=pre.fit_transform(X2,y2); Zv=pre.transform(Xv)
    m=lgbm(best,300,float((y2==0).sum()/(y2==1).sum()))
    m.fit(Z2,y2,eval_set=[(Zv,yv)],eval_metric=ap_eval,callbacks=[early_stopping(30,verbose=False)])
    bi=m.best_iteration_ or 200
    val=average_precision_score(yv,m.predict_proba(Zv)[:,1]); tr=average_precision_score(y2,m.predict_proba(Z2)[:,1])
    pref=build_pre(us); Zf=pref.fit_transform(X_tr,y_tr)
    mf=lgbm(best,bi,float((y_tr==0).sum()/(y_tr==1).sum())); mf.fit(Zf,y_tr)
    ho=float(average_precision_score(y_te,mf.predict_proba(pref.transform(X_te))[:,1]))
    return dict(val=val,gap=tr-val,ho=ho,bi=bi,pipe=Pipeline([("pre",pref),("clf",mf)]),
                vprob=m.predict_proba(Zv)[:,1])
res={us:eval_variant(us) for us in (True,False)}
lift=res[True]["ho"]-res[False]["ho"]; gap_inc=res[True]["gap"]-res[False]["gap"]
USE_SELLER=(lift>=0.005) and (gap_inc<=0.01)
print(f"seller'LI  holdout={res[True]['ho']:.4f} gap={res[True]['gap']:+.4f}")
print(f"seller'SIZ holdout={res[False]['ho']:.4f} gap={res[False]['gap']:+.4f}")
print(f"lift={lift:+.4f} gap_inc={gap_inc:+.4f} -> seller {'TUT' if USE_SELLER else 'AT'}")
f=res[USE_SELLER]; final_pipe=f["pipe"]; best_iter=f["bi"]; gap=f["gap"]; vprob=f["vprob"]
p_te=final_pipe.predict_proba(X_te)[:,1]
ho_pr=float(average_precision_score(y_te,p_te)); ho_roc=float(roc_auc_score(y_te,p_te)); FEAT=feat_names(USE_SELLER)
print(f"\nFINAL ({'+seller' if USE_SELLER else 'urun-only'},{len(FEAT)}f,n_est={best_iter}) holdout={ho_pr:.4f} ROC={ho_roc:.4f} gap(val)={gap:+.4f}")
print(f"(3b 0.677 -> 04-ilk(ES yok) 0.722/gap0.19 -> 04-regularize+ES {ho_pr:.4f}/gap{gap:.3f})")

seller'LI  holdout=0.6202 gap=+0.0996
seller'SIZ holdout=0.6115 gap=+0.0976
lift=+0.0088 gap_inc=+0.0020 -> seller TUT

FINAL (+seller,18f,n_est=300) holdout=0.6202 ROC=0.9086 gap(val)=+0.0996
(3b 0.677 -> 04-ilk(ES yok) 0.722/gap0.19 -> 04-regularize+ES 0.6202/gap0.100)


In [6]:
# Esik: ic val-split olasiliklari -> PR egrisi + OP1/OP2; holdout'ta dogrula + top-K
prec,rec,thr=precision_recall_curve(yv,vprob)
idx=np.where(rec[:-1]>=0.80)[0]; t1=float(thr[idx[-1]]) if len(idx) else 0.5
f1=2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-12); t2=float(thr[int(np.nanargmax(f1))])
def at(t):
    yp=(p_te>=t).astype(int)
    return dict(threshold=round(t,4),precision=round(precision_score(y_te,yp),4),
                recall=round(recall_score(y_te,yp),4),f1=round(f1_score(y_te,yp),4))
ops={"OP1_recall_priority(A)":at(t1),"OP2_max_f1":at(t2)}
k=int(0.10*len(y_te)); o=np.argsort(-p_te)[:k]; fl=np.zeros(len(y_te),int); fl[o]=1
topk=dict(K_pct=10,precision=round(precision_score(y_te,fl),4),recall=round(recall_score(y_te,fl),4),n=int(k))
for kk,v in ops.items(): print(f"  {kk:24s} {v}")
print(f"  top-10%                  {topk}")
plt.figure(figsize=(5,4)); plt.plot(rec[:-1],prec[:-1],color="#2196F3")
plt.axhline(BASE,ls="--",c="gray",lw=1,label=f"no-skill({BASE:.2f})")
plt.scatter([ops["OP1_recall_priority(A)"]["recall"]],[ops["OP1_recall_priority(A)"]["precision"]],c="#FF5722",zorder=5,label="OP1")
plt.scatter([ops["OP2_max_f1"]["recall"]],[ops["OP2_max_f1"]["precision"]],c="#4CAF50",zorder=5,label="OP2")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR Curve + OPs"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_pr_curve.png",dpi=110); plt.show()

  OP1_recall_priority(A)   {'threshold': 0.622, 'precision': 0.4767, 'recall': 0.769, 'f1': 0.5886}


  OP2_max_f1               {'threshold': 0.6857, 'precision': 0.5394, 'recall': 0.6672, 'f1': 0.5966}
  top-10%                  {'K_pct': 10, 'precision': 0.6509, 'recall': 0.4462, 'n': 822}


In [7]:
import shap
PRE=final_pipe.named_steps["pre"]; CLF=final_pipe.named_steps["clf"]
Zt=PRE.transform(X_tr); s=np.random.RandomState(SEED).choice(len(Zt),min(1000,len(Zt)),replace=False)
sv=shap.TreeExplainer(CLF).shap_values(Zt[s]); sv=sv[1] if isinstance(sv,list) else sv
plt.figure(); shap.summary_plot(sv,features=Zt[s],feature_names=FEAT,plot_type="bar",show=False)
plt.title("SHAP global (ilişkisel)"); plt.tight_layout(); plt.savefig("fig_shap_bar.png",dpi=110,bbox_inches="tight"); plt.show()
imp=sorted(zip(FEAT,np.abs(sv).mean(0)),key=lambda t:-t[1])
print("En etkili 8:")
for n,v in imp[:8]: print(f"  {n:26s} {v:.4f}")

En etkili 8:
  sr_drops_90                0.7310
  review_velocity            0.4319
  review_count_log           0.3195
  variation_count            0.3014
  Package: Weight (g)        0.2936
  new_price_margin_est       0.2572
  Brand__freq                0.2143
  Package: Dimension (cm³)   0.1661


In [8]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db"); mlflow.set_experiment("buybox_propensity")
with mlflow.start_run(run_name="04_regularized_es_ap"):
    mlflow.log_params({**best,"n_estimators_es":best_iter,"use_seller":USE_SELLER})
    mlflow.log_metrics({"holdout_pr_auc":ho_pr,"holdout_roc":ho_roc,"val_gap":gap,"base_rate":BASE})
    for fp in ["fig_pr_curve.png","fig_shap_bar.png"]: mlflow.log_artifact(fp)
joblib.dump(final_pipe,"final_model.pkl")
meta={"phase":"4_tuning_shap_regularized_es","framing":"cross-sectional propensity; SHAP associational",
 "method":"regularized Optuna(12t) + AP-monitored early stopping + gap hard-guardrail<=0.10",
 "use_seller":bool(USE_SELLER),"n_features":len(FEAT),"features":FEAT,"seed":SEED,"best_params":best,
 "n_estimators_early_stopping":int(best_iter),"holdout_pr_auc":round(ho_pr,4),
 "holdout_roc_auc":round(ho_roc,4),"val_gap":round(gap,4),"base_rate":round(BASE,4),
 "seller_revalidation":{"holdout_lift":round(lift,4),"gap_increase":round(gap_inc,4),"kept":bool(USE_SELLER)},
 "threshold_assumption":"A recall-priority (FN>FP)","operating_points":ops,"top_k":topk,
 "shap_top8":[{"feature":n,"mean_abs_shap":round(float(v),4)} for n,v in imp[:8]]}
json.dump(meta,open("model_metadata.json","w"),indent=2,ensure_ascii=False)
log=json.load(open("experiment_log.json"))
log["phase_4_tuning"]={k:meta[k] for k in ["method","use_seller","holdout_pr_auc","holdout_roc_auc","val_gap","best_params","n_estimators_early_stopping","seller_revalidation","operating_points","top_k"]}
json.dump(log,open("experiment_log.json","w"),indent=2,ensure_ascii=False)
print("kaydedildi: final_model.pkl, model_metadata.json, mlflow.db, experiment_log")

2026/06/09 20:17:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/06/09 20:17:05 INFO mlflow.store.db.utils: Updating database tables


2026/06/09 20:17:06 INFO mlflow.tracking.fluent: Experiment with name 'buybox_propensity' does not exist. Creating a new experiment.


kaydedildi: final_model.pkl, model_metadata.json, mlflow.db, experiment_log


## Sonuç
Regularize arama + AP-izleyen early stopping + gap guardrail(≤0.10); `seller_rate` final config'te yeniden
doğrulandı. Manşet = tuned-regularize hold-out PR-AUC. Eşik: varsayım+PR eğrisi+OP1/OP2+top-10%. Sonraki: thin e2e iskelet.